# GwenLand glcuda Wave 105 — N16/M32 prefetch T4 gate

Self-contained SM75 resource, parity, and two-run direct A/B gate.


In [ ]:
import base64
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import re
import shutil
import subprocess
import traceback
import urllib.request
import zipfile

BUILD = "wave105-n16-m32-prefetch-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "e9dfef911e99143d1d8fcf2090fdea73e8f4d83c"
SOURCE_REV = "29024366dcfd2ff15adfc6cfe95a449b36eb9f7b"
PATCH_SHA256 = "b5a11b0202b40cf9fe6ca9b68f19dff6b53e1eef39f6da60d32ac71b2bd6d9e5"
PATCH_GZIP_B64 = """H4sIAN1RoWoC/+19a3PbSJLgd/2KsjbaTbZAigDBp2xvy69en8eyw3ZPz4bsQ4NAUcKIBGgAlKixFXE/4n7h/ZLNrBeq8KAodd/03F4zHBaAqsrKyszKzMp6hdF8Tjqdsygn/uHZIliH/iHd+MvVgmaHV/4ltXsDL7aH3rLveKuUzmkenHfTjMzuknsvpldkHi0oWSYhJXavN3TdvSgO6Yb0dvx1u2Fv4NrjSX/Un8xoEMxHgT8I+/Zw4g6cgA7H80kvHISz2V6n0yGHIb08jNeLxd7BwcEdsf3xR9LpWT1yYFtOzyE//rh3cHj4gPwCRQD3AQmjlAY5+eiSMz+nZJ6kJD+n5MQeHr7pOySlZ1GW07QjYZLAj8MohLxdBomD+3geZYRuaLDO/RmQZpUm4TqgmQDfuaBpTBdkTv0smkWLKL8m9DIKaRxQksSL6y55lZMwoRkHFyc5oRmCirJzASyPkhhQS5P12flqnRNANDj34zPK8E1p7kcxDaHCbOVj2/cO9g7WGSVZHk6nebSk0+mrGGDG+ZFM4nScTmfr+Zym0+lTP7igcfiUvR6ZecI0usQ8X/HV8y/9aIEttcgzeL8pZebNzabT1+zhA81LGVK6grqm0y9jr+dlie/liTcD5M4oQy5IAFPy9ueP3vNXb6ZknUX/oOQxGU+GRzLx1YmZ5nrjoatSTz6+fV2kOW6R8svx+zc/vyvS7F4B8uOL9x/0lCLp/Yt3L44/aokDlfT81fsXzz56Px1/fDEl86GLJbsM6t7BHDgGlA8fvXzSYg0nD5FcFlkCAxf+Og7Op+Rlm3SekPc0Wy/yRwDAQiolKfDrp8WLNE3SJ3sHV+c0pXsHBH4voUT8Zp239GKtdrWUtXfwlRdBofZIFBPoeLz9RKTgj+PRav/7Ef92w/8gvt3sGpLSJIZGFxkWNAep8tMcmipEajqNk6tW+6haHyPqb6ru7UWL1dalC3+V0bDV7vqZl9Eg84BaQIYfiE2H5JDzj/gZcqG9d3AjOAAydkWjs/M84yRr/ZUGj9bjJxYRD22JHjbsSwatagHiQvoAOhe1dtGE7tJftb5F30irFXWvUn+1iuIzb7letJxJu/jgh2ErArwmo3abfEecwaCN2K3HOqggWSxAQyjaHR6SFxsfNNLcHhLnf3ZGgMzGAZ0JHAa9kS9oB7po5MfwdlHq+x+SYxItoRN1NUYFPqjISptavFGAXd9p6/jMF37usfZ538hpb9PrWQQR+LwN6daXzBI1aYSHVkSXPuqtrJUmV5noPYwJSPp533kiSY/YYZ67kbs/4oS1GWEBHukQ2xl0e21omO2M4KEO6wLFJRCu0pGeJptH4XXMNSfFvjSd8i6l0I3m5IGpCFttXchXaRTni/hBa/9UGKcOGKcOGCdlSD6DlifPfn5+TMC6RQE9Qq1/Di0jS7AS65SG+5K8+AMmr9MYO0Or3Ta7DrIZkQEmP2PaFQzGTOtAgCzTPt0onifdbAns/XuSWsT8FsVJ2iYPQFBGFhkYrRF1Aw1a+8pwpvTLGoxbRkAUP7wZDUQz9gFintTiKKwCoKnMwnS6SPyw9RBR0fF9IPJ2z+hyWbHrHo2R5GFLyjP2qr5jCdFRr2gFxEtjezJA7Ke/IB+8n96/eu48f2zL95OPr/7ywnbGxZenHz4e//QC3j8VwNhPQnjx5o0HrkNRQn7xwJnw3r1/8fLFx2f/8djeN8vX06yWDmBFMyRGpe27NFYxT3MpsnN/RcFtCJnXkVHsIgSwrWFkwUqpcjwQgGSdBmgSBapah0PRd0aKfTKvx3O29L6ZXHgJIKhUWeF/CfjCg1rHqsftGwZJuWW3o1TnKu6CnCb6OyKlyMRUIWKEgm5oto1I82aLJLjIvBVNvWzZqhDYAjU3BFW8jWhJEKxXfhxcky9rml7vQqw7I1al8xbMVObdUAOBL1PsEemTb9+qCOP3eiEH52Pp5w9aZg/bLxDgfj5UTsOpqu/x11LNN1ZR6eOv5fpvtA6st3tLl4GUC2/lh0Br7KrdMLr0AhotWmP0YMYad+IZOlaFfdaSdEMLmXTXRs+VY+M8zMufjBIlh7v1EPM95Fks6XRLXWqIzQYA6EZdNkmve3adc3dD4tBd0FjvTQcGTtXUDf8ENHH1z4p80kFoSATilUo68IEpxx9I4f9Ucthezx17g9GQ+2hsLCHbhN46jJGgVcYICZxeesVtl8XbbVDrirmSUK7rL6ALlQgiamn/ezdc5aleLAtqiunUqi3KjPl5noStK8ZPWZtCScuQBSqDkAyDy3O9fg+8qpbgSUONPMscYG5MQCUCtCo8bKbCxqQCq8LgsllEmskvaxiMgI/pfRlLxiBiGyTJBpvdhAMYzVrFnazzCh6mMFVxL5TFXUpXK4ei3wDCN13RGf7AculDQ0U/RpNWUnqcAuY3lI7SF6CL+WVTybOp5AHESl9KDomZaHpmZprmuVgVtXpTR9i7U8Yw9v9NycSfpPS0dBkutIAkYcuQUlNLNI76URGeJ1nuaSJ6SYMHpz2Q7KOSlv1cV1Bn4a0lGTJhnpyzjvOwUrtF6tvYUEzVbZH6xiOm8yiFrMsoW2IUDbA0atRNfU5Tw0v8R7RqPTQr0pNXSRah2Wx9a6G5aH8jfheNcMTiEjDumhWvpcEv+lId2yI4Bo5QBCJQmYXPVEL5yWPSu8U32odqOpSFGTRviPg5yisGF7+aMG/2202uzeEheZasYyDGzF+AfwVgriIcx5IkpiSKL5PA56HLhGTrFXTBLMNoa4BeJQnTaJ5r0YoWMksxdZ3xSFnBrTVzYlq9LkYlYHRfE3ASoboSBSRAcvCYh+SkeQDnsr7HKKkwuk0DiKZu9fsD+W1t0UbjySyjqfDkkKboBAnatUUQ7cjUKFjl4WOjYFmrNGdhsagVpeF6BbUZII3iSo+pEErR9iKYIscen8nXr5/UAEiM3T7tTz/tny28Zkvwad+CEfynYnyyc0llQySI/DylfphBSRwFfdoPr2N/GQUeDKtTwIi5hZDY47lLgwxzbAXZvt5YOlaNuW5u9q0ameD5rcpwiWeVvWUrZfmEBafrlZ8u1yuJFiq8TL6kdEX9XL5i00DOvDBSjYhi/Q29Li2vxn78Ou32zXYbn4XU8C+ugAAKzGMKDD7n6ZpCPlNlwXdQmcBNgMeLOghs5WdZDQV5VFz7wGLJ2rvoGdoXOVDSypyUPqBVs2q7rlWvFbTPotXaF22ioZoPtX41g+Q4eqDzlNJqoE0Wf6SXvs16iDkzVtYcTn8V8JBRGwD6VZ8dAQ5s6gyJCPIzs3ezF9bMYmZpcCj8O/EJOsJo4Amh7a7yTTEtuEPm32cOc+IGc5eG48AP+gMa9p152PftkU191w1CxxkOh3Q02z6HuQuy2hTmwB6IKUw1gznFYJ0e0DujyZLm6TWzwyRIcI4UWBTTTd55zbKK2dG9g+4ldGksM+r24C330zNU00tvNIBXPwzRWntsymvoomLuXkZZhBOcXRpjFbvpSqFouis/9ZekCwM+svIKD7v0XfnQ5vdNQ/5NQ/7r8ldsulf44+b3KK79jHpr7wDkEgNicUrPyLinptW6+N6FRobku9Uj231yVPOdESOjQRKDuypefUamXundri2OdJRqXbwiS9c5NbLP7CH57vzR8EnNV44CVicezZpm0NLv0kfu+EnNZ1Zg5mcU3E/xxrH10uSq8g0lxm6EgoayKDLL2Htj7otekfVCqykIevpLLQCkUnyWJqA/5eu5v5irF2yQeoGGGDBwGum7+SOnTEuQqu/S8JGr+Cw7YX9KwOYDu+RigQyVOkbT/UUEnhTO0cXr5YwiS89B3RMWDWDuUbcG+7NrL+dthKd4Jp9SuqxDCJM2iSUfs+LxOikhajvkaYdxiuA0As7c8EEAqIUOmxQioLsXUYAqnQcWjflEA8tZrKg4w1gSf8rWHN9ZmaqyFIqIyFo8xgUAFAqRjo+1LWb1ReGGt3T2RSICzxk+1xf6sspTmat4TObz2uyaVENmu0xJ1LtV1SoFgHAiy8UkqKClVu7WVPbO8xm/8cEWD7NejVC+A21X08Ehd1ZD7HceIlanVd55yySltfpmxiKCMol7saQLgnwWE6isOxujffBP+72R8/lI0ASaAYqSxoRN4W6IOyZPyQr4C2VXEViAWnCuhLbJTsF55tAEOGxvEWoWIe9bkJqdDm3XVUjZzlgK8f3QmpXQQoA4Ky+ASpzE+CbkloNZH+Ag8PGUGTkVBillcHiGLGjK0GcZNs0QXJ6hGcKAZbiuJjPx4BiCQWxI5whGcUMyRw8N5OcjLRggNeIyueSrEzAi8n1GYEAY+QtgbQeYQbKFPyOLJFkR9AbJWRqF3esueU+h+0bxmYKGAHCERbl8dZiLhtxLopgpWtSmwcJf4pIANg1Alv4FqDYKns21AhPFGQwSuHs0owtEAEAy3be4ZpUkaXQWxYAgVr+gHYYgGO08hTzdAp8eiaBSslwv8gg6P0nmKK0tRCPHNTrQ5SkZty2SJQwu3YBCQNzAGMXhmIWf2wpcIeEdFEvMJ+sEXYKLE4Bw6/k8CiLwt5g+8Unqn50BCWAssSDPPh4L5IDegjHKfgSQBcgq2JedL2oNDD4Nj4owI2/lYyJKkx+Y68eqWC+6V1FIRT2a5SkgOkeC4PkYOhyyGhRsRnMOAdrYzZR0q/85IIVnqjWEmz8QxcERMX9QC+uBhI+OCYyOeY3NyGYlZHkFOn0Kk5qZJtU50upF5cTr3tJA1yr+51COmjG7LmFm1yN1rRv3GqRE9G4LVgOr+N/wEcBwdzNFdhA+i/dxiZVE3t9Us6lHlSuKRa6+kT50j8o8ZKo5ucJY8gy7InQvEGqpUAqpRkLmIJCbipiwxrg1EsLgX/kpKDEYmsKAbSTIEYeiJwxF0b59VFN04ce0IpOmawl1q6KiMjAwUYotylgE9MQelms1/FEdAoPxBhDt27i8E9yGbnfYr3TfigfLYQ2OyjQbQY64nmhji6fXE20eoZLlZg7bxHsX62jlOiZK0WyUiITdRSL4bzNxhkyuZVKvopVs5vlgfremq8c94U2BfCgqKz16khDqp6DKQd3mUzLzUzaHAY4ZDbn/fXWeSFXRJSvbBgVXwDr3s0Ilg+IFM8CCSeQEDD6o2yUo3yMGhg/k1EcsSaKcCTD4fKj7z9Z+GgqdDB1w1V3knFIr0UBb79+sZ9YMtDCXXBJgQjHHk+ViyhIXnLYdi0u6UyfjjBevngM5UNrJoZwYL6TV7ovy/bryeXQmy34ny2p1u03am/EU1zlEMVvjQFqvhSJvlynjOEIVjY7KuPEk/L+3ecl/46MCvpjqHWPA2TC/Imp9meMUTPdskcz8hXSamE5Q49L6PCOWx9maZ8wN3NY8E24jtubhPSgc6E4WXyG64quooac+ZWsqpuT0BH1UfP58iuMRRtHPp6Bj4Z0tUfhc+DLHuG6nw+PXqHPR9cjWS8r1lnDUULUcQacY2E5dVsNRm68XC+Yko07vko/4iflUiBD6Tf+gaSL97xn3u4JkgeNmbEe3ouiKIebE9FHYrBOgwdYFC2cP/bATVnVFZ2vjUwGxKkra0FXksZ1Rk8XWB6AFdNutsdrG6FQVs52jimnWsw71oW0d1KwO6rgOaJFzVBojF+K0ihbJ2ZpxY70EwwXDIYKrtHF7AWq7ZFneZMB2NTwVk+4VztmsKtQcVUVnjzU1OKomsz6BmapFXVemjY/qzTUMNLjm5i1B7aLpcGLrAxWMjqJ442TX0o+viZjKsbgwMzCsO/HPYkRzlUDHYQKuID1NABI47XE2xyEJd9wJG0wCrbAbJ2A+xUwbjhRwEpSmYnwkBp96nKUQXUc6N7XKm+PIYw3YWXE6MKzT4Y5ydCpm12Gscur0O3rYDDYqDuFQ8pVd4Xdu+4dxo3kq4ptYJ9Cw1pJxw8Jb6DhahxShCJYu/tfB1pvOUlQSwaqVc40oNgY16/Apldv23mtQHE5Pa3K1s/JkYTmcXmEW0BYoGBzBUWNx+cdu0l6l0FYjDWq0SU1RgW5DwOy2wrUfDdTBlRR9YSyJ544rMiASxwZtCi/VAa2CYaumchNeuq7aSny7nl41ON1aUn10RrsV5ng2Rdyhv+KyOUyI4rOpUlw9HEUQXGivjVpRU8j4ZLBOU1SP3D2u7auyM6tFAYVy6au0fm1/YzjzTLXdnM2D8P8nRxX3MiSP0LIfnzxnaD8qvLqGLuYIbOrtMU9mf+o6IE+diDxmVIS7ZYzEIKmk5YeXuN4lIwfu4cWsXVasfVup7rI04nJ9FvascJ0X6nOP1y5TQ5gKxAD5TlpsnNY2luIIUwPG5WkhCT44ZuDsmaYMvTzNSCkYwfk6vrDAgoJXhREpPUwGTheYWCCWFljtVkQXg//GdIAzaPSkwKxYRZl+vV9UhOt1b6oI3ZdUpIJapyVNcHUTAWb3VzMWDZpHn9JQWasaSM5sYFx5K4xiAgS/1QzotOmSUckx4LFzseWLRQvB/4EvLKJSYwl5sF91GAH56HaGujvys5ad2uxLVstOfQigZoncwiPUqZoVZM1qENcmmTJtaskgLfqO4dLP04hHLFdpgntz02zKfEqmQEdsio+HdJnkcyeQEfl1zyoAsQLjbtceFCUyfGDZ/Zy8tsHXQi+QZ7WHLNzD19OQS+iQoQLGJuKJmIanYkiFk0kiagG+K24eFvuAM92dxHYgbbu1wSfBzqE2ECpopmXYNllq9JJ+3zKKFv2kiCMMRZXjauiHJeH/VZ+fQ+b/28OGZG4X+/3CEL55c4zbdNaLXNA5A8lecOpwzVeaXGIRLhkNgTYU7v17yiM7+bnPJ338lJHXn+NM3xonDFgdD0dETPYqaZKS1N2ZyGpgqBkS16StUyWCq1kNV/MG3EKWuTR+PekdgrgS/IP0Phnz1zG+3nAB4xKkgJyyQbWKH/CBCpsFWWCH4nZhdX6dRQEO9XEYtEoyNjPy9PS1dWJEHOIk7hRZmEtSTJvIYZEQZE5pZCQff6QFIXUFoVNRBmarTkppBUK95BdqTj4UpXaAqMRT09V1Cx/Eq30vBErGqdBpTf2uime1A94ZzaJW+VRFVClpe9Bo9mRxe1CAO7ojj7eQT8UXDItSolkVYae/xaDI0k6/ukyhHM7NCpLKgkxItNnSdNVZxxEureOhYx58SemCO3F5UsSpwfNC5Y96v9ZJN9cKmXqlXzOe1dYSWTI+rQPZ1TQ0DO+tUon+rvCKfmQAdO4N0HFrAfbrACrWvEzWoH14NDXDQFIdHyw+hrpK9DBTpk3J8nUbOLrpzeXKwSM9xTZSKgWdxoL97QXdxoKD7QWHjQVH2wuOGwtOthZ0GonjbCeO00gcZztxnEbiOGXiVFQDIludMxMyczFT8xiELVVgYzNt0RDaYxZqnRJ+moyK0vPgOFsNwUOt+koifY5fXzKkN61IsM2EYulQA1FUyVkZpFpdVHxnXehMOPxiIZElCKMNuX+UaWSW+mjEvXcvvXfe87cnL2SOBxjcM5OP7WKtiTkLwtt7ysJbuN5EKzIt4OkhORPy3z5sgWxzyOVYlFHP3z5o9dg9E/rTCnCd6Axrx4D2dNqIzUy0kw9OP2vUEsO1RopqwDTGnaoxl4EBlpuidOKX1395+/bdtI7Bk3reThQSv7x/9fGFrjc/CGeZqoXGONsjuoSQ7mJuARVpTQS8JBzs2IVCOrJcLNliVDvFyN1nuX6uJxopi9wmHTyfEo8K6EqwTdVkGzU1yQdPfVoBjwJyikEehCcX9mkFpk34sLDA5/IqwTr5EJCO31fbBvJxqgbCCEyuITTKCRzUdHpP5/OrLFtTTX9dHABRuSOvxxSRwcLJIFegE9XIantIxu1NhrfFDtijMxhujYT3nXtHovvNEUL8U3VA+YJL2WPsWzRmkXu71ny5XWu+vLvWfLmb1nz5W7Xmy21a8+VdtebL301rvryn1nxZaE3RCTq4tWNF0w6LyEg5b2UX0WqFOyH5qpNrXJjVSeadlJ3XxlbUtFlZDU3NG1e4vv7wnyfPFJ48qMC6YlcEbrsbt7scx2PZsWdytRH+vsqJPz7HV7es/sbiegDHJto6T01NfHduizyZngkjrsCtLq6HBbcJNYjdUN7Ryx849RDQFz93GiCoDQyn5sCqHpS20aG3FaBdAdiEnW3unShCQ+s4xvAPB95ZUui717iACCMuuDsZZ0zB5i07bAlEj1AffL4LY6JFmxYYiChWrW95zMYXLBpYKTgUkZ/agnK6oCy2uAVhUaDWYsNPvjLDkEZdUhuE0CkLIZM9vvhnwGWsP/hcnQcZl/xN/nVS91Xb/NGcqHu+S99EEjG8sId4yhieBIYqu5uN2b++Y3SaPpsT7E8AcdYM8XfI/6rU315R/YYWs1rZa2+sXYr+7o0fiL+jf27jB0bj7Ts1vujs3Jy4XP6GetdOY9a7mdmeM+3YHzcliyZr8TeeDEnYYZnym0sLPgeCqGQ2qcnA85x2rw4Idl6m/xqAoPJhSPCctr21HYo2W5tT3j1VbVWvGaFxqVXjxlbZzUAmpVZNjrbqJ7tkLu1aG6mFxLkuxVUg46qjJhTmUPPx/tRsf2q2PzXbzprNKekA5z6arV/SAf0/WLM5JX3t9O6h2ZySvnbs7ZrNKWk250/N9qdm+1Oz/WGazS1pNvc+mm1Q0myDP1qzlfS149xHs5X0tdPfrtn6Jc3W/1Oz/anZ/tRsf5hmG5Y02/A+mm1U0myjP1qzlfS1495Hs5X0tcN3WylVNS22yuBWI1w/tqQ0zwiuXdJnPSpTXriI9iqN8ExcIQY1cyqlJem98gyCUps4TacwYzNvBWZv9RDzCV/KxrebRLE4NFmsWsAVVuc4IR0nYsdw95Y4tBYuRxpgEJNt+WO7XHH1gx/+3Q9wukfuIIpiMsNtMTXLIjSV3qtb/1bsbMFVdnLhhVxxodbDGrMqfH06h9cvJlO0GYdf7LrtqWK9NZ/f5JuLGla2yjXZpXkfsZmdzwqJrXrFvpNcTi1cOnLCD1I/ozYRwSG70JIGXqMSXm7TiluxPnJUj1ffwEujjLaNtAHNvkBzzMMzN2Ia5Bc5c1RmI/4/ProfczT5+pdhj8OHpP/i7OFDVkdJ0X93rrjcnf5X54rD/eP/X7giXIF/da643LbfKBMqpnPFOYdH/F6Y2w4gXCahcWlaNU2cJDifjye9mRvYwSTohQM6sp1e3x72Ro4dDobheDDu+eP53O12Q79P6Www8Knd79n+eDhxHDegQT/ou+PZfGg7k1no+nN5UiEOtbbg1nzEoEjHEwWBI0NyAP9PCLyu1uy4GzC67z7+zcP7XLxfjv/6YjCZkodZnrKN+sFiHVIP3h609stnFQ4meFQh3lgD5vtQHH2DR4kBFeGBphHbgJsl5kZivsMBsFrDeBF9lf/x6iMuebvmR1B095oQG493RGw8VojhSrxiKd734KRkycLPjdtGGm97Yws9X7486YTg9IAn04AXOxhsJ8TE6Y7skh9OtI9id5Y6a4PVSRcUSde5ijJOIvA+Oyy5A+6kONRIHn6+J3B6+pe3z/ACNGjTY75EhhyK5caEHef4f/7X/2anLhWnRcrNgAgeVyh0cFkDemA/vXjz1y4TmbGLIjMeWTZepEfmMcGup65QYTfJtMQ5sFN+Kw0/Bla84Na9Kb+gpvME/MNkQb7uyWPSMR95LC5yIw8fiqL4aTwZ4gd2vNNjdp3bHrkp85NvWMv4jhjmi0cxipjkGS69/Dtl5OK37dFFNMO9chTysxNADtnxHxnFU66A4dkVHhHGd7+zSsZjcPUDH++x0+/PYTfmCAcVhgNddrFU5X6Z+xHnoEQcQYmCNIpaOnGYGmMMf85PI5ZL7MQCBWQ1IoY7Yvw8B+mChkzZthh+tBHbjQFSiI31F+xQJQZutVhnYnf8Rg5xNCnMgpRdPUjes0NjcWXErycA9VdxKJePG/L5FicGDpe8HiLX5ngkV+Cv/ABvRvTFHXGE3b1FACbQFbffxAhhHePiCpDhkPB7C7ls2lw47UlfSCc/8Ytxbjx+wzWMkDZdaqfiMiqrTqQiTUeA6ooEmzWFpg56xLW6Sq2hFguSEGVBQwMgKjwYYz2u+KaEfxZhhbLoFBgWfH0b890zyFMOkrEGj8WG4R0/ZVLuAeSnY4iTGsh/iBRUshwWYwi5OseQ3a8CpV/JIsJhIqoJthmHt4KN9yJ+EFmxLdTPhZ4WudTtXtBOZI3Ts5E1DrgJY2lrarLyWxaYLZmStyuUqEe/sFdOnifWnljScojJHRgZym6JXkSHqTPQ5Nvv7FS1oAXRapEy8kTt5TNrQXG4o4mQNTGboFWl5EBv0S/nlIm57JfZebJehIRdu8ltJm7gbeGRLdAd26iV37Bz7fBIlmSOWQpg4vAW6HrYFzEDXjTq53zHFdNov4oLyt4DxF+P8Dy8ovjpr9oFbVivvHDt18+8szlOn3HU7d3OUX5zWd+ZMp1W02JTV7PzpA/XK3EvmSAD6lzJ7OL8Yg2+0aN5RQfbK+JGob4WvlJfHZNsVGN2zXKbWOnBMWoOpQ3QaId07q8X7BSpiHXLQGxzQz6Ja8RmfnChwYIeCdKFe05w4yFovuPDp8Q/w0MBc+yE7HArQQTcXOoxqKHEifl4I6YND/oTx4IhBHyKlqtFlUfsIG7wNDKqf2FHmIMMaXmO9jrqHk8oXqE9eVxk0KmGFopfpxhfTqeXfuolWWu/fEeevB8PDwnPvAwck5Z+BWJDjb9LJUSvhKsH9ODmNRWWKMSuN+E6mN+i1sVOJvRoq+S04tHrnXLpbQTkMwsICsaIuTcHu8guEdlyS4EEtW/cG1Ff2W+CbSBJtSsvuav7WfXYeqXMNJhQLPvtErgPyJc6263/lP1cCvvJZB78U9BNbs+17MnvIvL1AqjrAeDawf3FXr8acovoC1NSEksDja9Vfu8omgC4UVzMlt5VZPTSVZFsFBtsafOwrCw7B7WyU3G4tgpPJUcZ+1KWG+2WHiVJBxVJOtAkySCtprOBqo1icvzx44n3/u0vH7YoLKC66ylPHnm09IvMW6WQgX/z5ti9DTww4YteB+9sk4llu9DbHPg72N7bGNHJ4yfEtozuVSMLpg6UktGRZ+Py20mYAwoiiBcYyHtGzr74Hpv5UB/AKXRCdWsJ+DLFlSTg8NrOWL5zkVXlRP8y3/HmGDNduw+GJwCFYk9jrYbZyAvO0XobWZG2lQ+M2NWv/mVxdYlBISACXnnnnS3W1CKKCBbhzbeY84hDTN5ki/DGWkqNWMpNs6oWz9JFlYGXLbFKkmfViYrx8RIVhEoq3dH5/xKfaxXwP1sIDv6ZQlDb4v+LsqEa1q7VEYW9EFcudUSQoxLZqHUy3l60GrUU4GGZX/iwtObjeFy+JpDbnVJWRnrzk+CK+dHgIdOwLvAF3Rl3ZI1u16+KlXWf8d7A2txNFq6e46WWaewvQddkoUJfXTB4/GbUY36bPeqDKbG3NzWji3m36pljGKwxICnDbTzS1ubAWMinPE7Uwz94yznNcjGKY4M3fmUG+iUiTBWx6e4sSFZy+yqOh+cx2X55+UNshUXuFRKsEsFw0gQhmgOQZWKoO/3U6BNPgmGEcEZaeIXf+gNCqg5wZQGK9PusfMOzGJgKSmg3gZcv2mZkYA0UIRLO8ScVfqNP42dQft5q462IrW/cc/smPdLC6WwLkXKESA3d3URKTBhsrUWXsFopktJTQyt+b5a6PpxF8417qU352fmy8i00PKg0ECcebm2hXl2deBQxFZB+0Q/YnTLsshs/BbXAJYbFS+O8LBDV++xru0PzjIF00FV51r7CuClBRxAWb3229AK8q5KLh2PbLo9Lgm4e3CIe2zmtNMK20CP5SUy1WMaOdy1Q5adRfg5ZooCJBpXH8vJ7PY2JMzk0YqQ9UpMeBSx1Jw2fD+2wyF/E9myzyzPkoWFC4v7tFO8kumoFi2i1up5O8yTx8FBcz0/P1mz1TvtzVbftfM0uZ6526R0Y7il5+My8eJdfSX3lfcmm5NnPIb2MAoqbvKs5+BF3Tbk2WyBsbil73ZRgCGbxWVfYxVclt/IiPhTf9+w8skettkV+WrzAuQ2jg4Z0tj7z/Cyjaf6g1azdlQGp1+X6aHgbSNVxjJ7SLkcd8DJwLGh6BlKRlL5KtVL6fAclo8okF16Sejiqbn37Juk1nb6Iz0D+W/uq77FIAHB0IS6exSu3cZ6GV7Mvbxs0Ag/qpll2VTp7wLvB8WEjv2zkl+t2ObrTKuTUKkmkxWTPUlJmQfG6ihMOO+J/YnapbT07S6UxO7uFB+9sPS3dYy2ahMfQ/YDPnnoKvMskCq26/Lg7+A75N3eEv7kj/Os75U7ulDu6U+54l9yfj0yl1uW2sHTD+LwEvxXQaAFMviyYPnRBLajP3Gixb3a7XJgN4+yalF5dM7i01NwYbphz7E6DCTkBg7Fhlky34ZoNw4lHsK/mXQA4717A0s+LZSdf4lke8moasZohk0clVgwaQDTnQBY4r7ZOxT0kKcVJGraNXjvBUR3w7gdpkmW8wIk4u4tPWfWdIU5WHfT7ciQFGoLk4Ntn0s7/2ym+fuYvwv0aj40hBjhcdJV5gLknJ4b5NbQeVuXJwEWrrfsOUv+W5gK6/PYCD7VWa19dxTnsDj7FxlWcn+J9VMudW+CZIaEu4oJjLwS966WdynLvG3agqQG/vYoSjTz65UHLHCuW62S+EM1a33/9vt1l3hzY0x2L3BRFhExM2ACh3x/3LbdeKPT2P6gZXbIFGBYuzMBz5vqqUcpTlFJ1oEkVOvWGSS/EqlhZUiNS2zmCjr3GElOEyre5CqGqQDQ1yDb497sKdl8z9NX6uQBsR6FeAnYso4vAbXg0QtkvbzTZLxAhMPq8KzhtM4AOybkXfarzGEVFu26Y2m/XQKkntfsb2XkPdN3fgK69Dd2t/am49XIH5VgpKq64vE9Rca3lvYres1bttuM7lxXHTt2znH2/crOG+h7cWtC+e8Er7P53Lvb9p973dy+T1pdpiO4xKyQskuO69dU1lDUN2d3KmvX29cvd612rNFlRL8dhLIxBsbt7ZwuwfAFGkpfrhW86UGzYFCQwmsoiNlbSirdGgDNo3Z6Hd7L3oOam9dvpOo5pWlm6rT6LVdtBMBq6tDfrz0YT1/XnbKH2yJ30aOhQnzqDsDcfzSfzbpe6s4k/GNKZ70MeSoeDWX84Gg/C8cx3wuFo0u8P5vPBqHHVdlF1ZcF2kcQclSGbBMA/zkgsvWXGFqPvrT3S9EPC8QHql4zdDgSP3S9ZN8QLLA7Il8zjt/oc7QJCXCsgwfBXCYq93Q4tmpOLIgQhAxltGbAWTgMXKmwbDoZn7brp+2aotbESBczC0fYt8PiGBx/jcOX1Gd7xycnbn0+evXg+5XPqaJqm07e4vu7x7VA5ZKPYdBrTq1Z5DUPdrxmXLgZAPFzkh8GSr7vhoU2375TfmKJFavPp2ZWf47ynnFjtAAs6wIJOMSdqfdoXw1w5m8kjHMUMbXLRMKe57aezdLdSuxD5ZpdMKVvfLIVu5yjotl9ghkK3/WSPvlN2EY/archmd+ibu0G+3jEfY+5uWVECdssZ75DvNvbf7JHdVdGtaojcQQ3tpoLIb1FB/wW0x4ptQqIAAA=="""
ROOT = Path("/kaggle/working/wave105")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
TARGET = ROOT / "target"
FINAL_ZIP = Path("/kaggle/working/glcuda-t4-wave105-n16-m32-prefetch-results.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)
RESULTS.mkdir(parents=True)


def run(cmd, *, cwd=None, env=None, timeout=7200, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({k: str(v) for k, v in env.items()})
    p = subprocess.run(
        [str(x) for x in cmd], cwd=cwd, env=merged, text=True,
        capture_output=True, timeout=timeout,
    )
    print("$", " ".join(str(x) for x in cmd), flush=True)
    if p.stdout:
        print(p.stdout[-12000:], flush=True)
    if p.stderr:
        print(p.stderr[-12000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p


def save(name, p):
    (RESULTS / name).write_text(
        f"RETURN_CODE {p.returncode}\n\nSTDOUT\n{p.stdout}\n\nSTDERR\n{p.stderr}",
        encoding="utf-8",
    )


def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    digest = hashlib.sha256(FINAL_ZIP.read_bytes()).hexdigest()
    print("ARCHIVE", FINAL_ZIP, digest, flush=True)
    return digest


def fail(phase):
    (RESULTS / "FAILED.json").write_text(
        json.dumps({"phase": phase, "traceback": traceback.format_exc()}, indent=2),
        encoding="utf-8",
    )
    archive()
    raise RuntimeError(f"Wave 105 failed in {phase}")


def resource(log, entry):
    marker = re.search(
        r"Compiling entry function ['\"]" + re.escape(entry) +
        r"['\"].*?(?=Compiling entry function|\Z)", log, re.S,
    )
    segment = marker.group(0) if marker else ""
    number = lambda pattern: [int(x) for x in re.findall(pattern, segment)]
    regs = number(r"Used (\d+) registers")
    smem = number(r"(\d+) bytes smem")
    return {
        "entry": entry,
        "found": bool(marker),
        "registers": regs[0] if regs else None,
        "static_shared_bytes": smem[0] if smem else 0,
        "stack_frame_bytes": max(number(r"(\d+) bytes stack frame"), default=0),
        "spill_store_bytes": max(number(r"(\d+) bytes spill stores"), default=0),
        "spill_load_bytes": max(number(r"(\d+) bytes spill loads"), default=0),
    }


phase = "bootstrap"
try:
    patch = gzip.decompress(base64.b64decode(PATCH_GZIP_B64))
    got = hashlib.sha256(patch).hexdigest()
    if got != PATCH_SHA256:
        raise RuntimeError(f"patch hash mismatch: {got} != {PATCH_SHA256}")
    patch_path = RESULTS / "wave105.patch"
    patch_path.write_bytes(patch)
    (RESULTS / "source.json").write_text(json.dumps({
        "build": BUILD, "base_rev": BASE_REV, "source_rev": SOURCE_REV,
        "patch_sha256": PATCH_SHA256, "patch_bytes": len(patch),
    }, indent=2), encoding="utf-8")

    gpu = run([
        "nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
        "--format=csv,noheader,nounits",
    ], timeout=60)
    save("nvidia-smi.log", gpu)
    fields = [x.strip() for x in gpu.stdout.splitlines()[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"Wave 105 requires Tesla T4 sm_75, got {fields}")

    phase = "reconstruct"
    clone = run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800)
    save("git-clone.log", clone)
    checkout = run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=600)
    save("git-checkout.log", checkout)
    applied = run(["git", "apply", "--whitespace=error", patch_path], cwd=TREE)
    save("git-apply.log", applied)
    diff_check = run(["git", "diff", "--check"], cwd=TREE)
    save("git-diff-check.log", diff_check)

    phase = "ptxas-resource"
    ptxas = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    if not Path(ptxas).is_file():
        raise RuntimeError(f"ptxas unavailable: {ptxas}")
    retained_ptx = TREE / "glcuda/src/kernels/glcuda_sm75.ptx"
    candidate_ptx = TREE / "glcuda/src/kernels/glcuda_sm75_wave105.ptx"
    p_retained = run([ptxas, "-v", "-arch=sm_75", retained_ptx,
                      "-o", ROOT / "retained.cubin"], cwd=TREE, timeout=1800)
    save("ptxas-retained.log", p_retained)
    p_candidate = run([ptxas, "-v", "-arch=sm_75", candidate_ptx,
                       "-o", ROOT / "candidate.cubin"], cwd=TREE, timeout=1800)
    save("ptxas-candidate.log", p_candidate)
    retained = resource(p_retained.stdout + "\n" + p_retained.stderr,
                        "gl_gemm_mma_q8_bstage_n16_m32")
    candidate = resource(p_candidate.stdout + "\n" + p_candidate.stderr,
                         "gl_gemm_mma_q8_bstage_n16_m32_prefetch")
    resources = {"retained": retained, "candidate": candidate}
    (RESULTS / "resources.json").write_text(
        json.dumps(resources, indent=2), encoding="utf-8"
    )
    if not retained["found"] or not candidate["found"]:
        raise RuntimeError(f"resource entry missing: {resources}")
    if retained["registers"] > 72 or candidate["registers"] > 80:
        raise RuntimeError(f"register gate failed: {resources}")
    if retained["static_shared_bytes"] != 9728 or candidate["static_shared_bytes"] != 9728:
        raise RuntimeError(f"shared-memory gate failed: {resources}")
    for row in resources.values():
        if row["stack_frame_bytes"] or row["spill_store_bytes"] or row["spill_load_bytes"]:
            raise RuntimeError(f"stack/spill gate failed: {resources}")

    phase = "build-test"
    cargo_candidates = [
        shutil.which("cargo"),
        Path.home() / ".cargo/bin/cargo",
        "/usr/local/cargo/bin/cargo",
        "/opt/rust/bin/cargo",
        "/opt/conda/bin/cargo",
        "/usr/local/bin/cargo",
        "/usr/bin/cargo",
    ]
    cargo = next(
        (str(path) for path in cargo_candidates if path and Path(path).is_file()),
        None,
    )
    cargo_env = {}
    bootstrapped = False
    if cargo is None:
        bootstrapped = True
        rustup_url = "https://sh.rustup.rs"
        rustup_script = ROOT / "rustup-init.sh"
        with urllib.request.urlopen(rustup_url, timeout=120) as response:
            rustup_script.write_bytes(response.read())
        cargo_home = ROOT / "cargo-home"
        rustup_home = ROOT / "rustup-home"
        cargo_env = {
            "CARGO_HOME": cargo_home,
            "RUSTUP_HOME": rustup_home,
        }
        install = run(
            ["bash", rustup_script, "-y", "--profile", "minimal",
             "--default-toolchain", "stable", "--no-modify-path"],
            env=cargo_env, timeout=1800,
        )
        save("rustup-install.log", install)
        cargo = str(cargo_home / "bin/cargo")
    if not Path(cargo).is_file():
        raise RuntimeError(f"cargo unavailable after discovery/bootstrap: {cargo}")
    (RESULTS / "cargo-discovery.json").write_text(json.dumps({
        "selected": cargo,
        "bootstrapped": bootstrapped,
        "candidates": [str(path) for path in cargo_candidates if path],
    }, indent=2), encoding="utf-8")
    cargo_version = run([cargo, "--version"], env=cargo_env, timeout=60)
    save("cargo-version.log", cargo_version)
    common = {
        **cargo_env,
        "CARGO_TARGET_DIR": TARGET,
        "CUDA_VISIBLE_DEVICES": "0",
    }
    tests = run([cargo, "test", "-p", "glcuda", "--lib", "--locked"],
                cwd=TREE, env=common)
    save("cargo-lib-tests.log", tests)
    summary = next((x for x in (tests.stdout + tests.stderr).splitlines()
                    if x.startswith("test result:")), "")
    if "66 passed" not in summary or "0 failed" not in summary:
        raise RuntimeError(f"unexpected host test summary: {summary}")
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave105_n16_m32_prefetch", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)

    phase = "direct-run-1"
    env = {
        **common,
        "GLCUDA_GRID2D": "1",
        "GLCUDA_NTILE128": "1",
        "GLCUDA_BSTAGE": "1",
        "GLCUDA_GEMM_N16": "1",
        "GLCUDA_GEMM_N16_M32_PREFETCH": "1",
    }
    exe = TARGET / "release/examples/wave105_n16_m32_prefetch"
    records = []
    for index in (1, 2):
        phase = f"direct-run-{index}"
        measured = run([exe], cwd=TREE, env=env, check=False)
        save(f"direct-run-{index}.log", measured)
        direct_line = next((x for x in measured.stdout.splitlines()
                            if x.startswith("[wave105-direct] ")), "")
        resource_line = next((x for x in measured.stdout.splitlines()
                              if x.startswith("[wave105-resource] ")), "")
        direct = json.loads(direct_line.split("] ", 1)[1]) if direct_line else {}
        driver = json.loads(resource_line.split("] ", 1)[1]) if resource_line else {}
        records.append({"run": index, "direct": direct, "driver": driver,
                        "returncode": measured.returncode})
        if measured.returncode or direct.get("pass") is not True or direct.get("bit_exact") is not True:
            raise RuntimeError(f"direct run {index} gate failed: {records[-1]}")
        if driver.get("retained_active_blocks_per_sm", 0) < 3 or driver.get("candidate_active_blocks_per_sm", 0) < 3:
            raise RuntimeError(f"driver occupancy gate failed: {records[-1]}")
    (RESULTS / "direct-results.json").write_text(
        json.dumps(records, indent=2), encoding="utf-8"
    )
    verdict = {
        "pass": True,
        "runs": len(records),
        "minimum_speedup": min(x["direct"]["speedup"] for x in records),
        "all_bit_exact": all(x["direct"]["bit_exact"] for x in records),
        "resources": resources,
    }
    (RESULTS / "verdict.json").write_text(
        json.dumps(verdict, indent=2), encoding="utf-8"
    )
    print("WAVE105_VERDICT", json.dumps(verdict), flush=True)
    archive()
except Exception:
    fail(phase)
